In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os, re, functools, json
import pandas as pd
import numpy as np

from src.scryfall_http_service import ScryfallHttpService
from src.scryfall_data_service import ScryfallDataService
from src.cache_service import CacheService, CacheType
from src import image_service

pd.set_option('display.max_columns', None)

os.getcwd()

'c:\\Users\\tallo\\Documents\\programming_projects\\mtg-coding'

In [ ]:
def get_commanders_from_txt_file(fpath: str) -> list[str]:
    with open(fpath, 'r') as f:
        return [x.strip() for x in f.read().split("\n")]

def get_json_file(fpath: str) -> dict:
    with open(fpath, 'r') as f:
        return json.loads(f.read())

In [ ]:
sfds = ScryfallDataService(
    base_url="https://api.scryfall.com/", 
    json_cache_service=CacheService("./cache/json/", cache_type=CacheType.JSON), 
    png_cache_service=CacheService("./cache/png/", cache_type=CacheType.PNG), 
    http_client=ScryfallHttpService(),
)

# sfds.clear_caches()

In [ ]:
cards_df = sfds.get_cards_from_bulk_data('gameplay_columns')
cards_df.shape

In [ ]:
def get_supertypes_from_type_line(type_line: str) -> list[str]:
    supertypes = ["basic", "legendary", "ongoing", "snow", "world"]
    return sorted([x for x in supertypes if x in type_line.lower()])

def get_types_from_type_line(type_line: str, valid_card_types: list[str]) -> list[str]:
    return sorted([x for x in valid_card_types if x.lower() in type_line.lower()])

def get_subtypes_from_type_line(type_line: str, type_to_subtype_dict: dict[str, list[str]]) -> list[str]:
    subtypes = []
    for k, v in type_to_subtype_dict.items():
        if k.lower() not in type_line.lower():
            continue
        for subtype in v:
            if subtype.lower() not in type_line.lower():
                continue
            subtypes.append(subtype.lower())
    return sorted(subtypes)

In [ ]:
def drop_all_na_columns(df: pd.DataFrame) -> pd.DataFrame:
    cols_to_remove = []
    for c in df.columns:
        if df[~df[c].isna()].shape[0] > 0:
            continue 
        cols_to_remove.append(c)
    return df.drop(columns=cols_to_remove)

def rename_bool_cols(df: pd.DataFrame) -> pd.DataFrame:
    fix_col_name = lambda c: re.sub("^", "is_", c) if df[c].dtype == bool and len(re.findall("^is_", c)) == 0 else c
    df.columns = [fix_col_name(c) for c in df.columns]
    return df

In [ ]:
def are_multiple_keywords_in_ability(ability_text: str, keywords: list[str]) -> bool:
    return len([k for k in keywords if k in ability_text]) >= 2

def split_oracle_text_into_abilities(oracle_text: str, keywords: list[str]) -> list[str]:
    abilities = oracle_text.lower().split("\n")
    arr = []
    for a in abilities:
        arr.append(True if re.search("^([a-z\s]+[,]*)+$", a) else False)
    return arr

def is_keyword_ability(ability_text: str, keywords: list[str]) -> bool:
    for k in keywords:
        if re.search("^{}".format(k), ability_text.lower().strip()):
            return True
    return False

def parse_oracle_text(row: pd.Series, keywords: list[str]) -> list[dict[str, str]]:
    if not isinstance(row['oracle_text'], str):
        return []

    abilities = split_oracle_text_into_abilities(row['oracle_text'], keywords)
    return abilities

    return_array = []
    for a in abilities:
        if is_keyword_ability(a, keywords):
            return_array.append({"text": a, "ability_type": "keyword"})
        else:
            return_array.append({"text": a, "ability_type": "unknown"})
        # elif re.findall("^([a-z0-9])+:", a):
        #     return_array.append({"text": a, "ability_type": "activated"})
        # elif len([re.findall(f"^{x}.*,", a) for x in ['when', 'whenever', 'at']]):
        #     return_array.append({"text": a, "ability_type": "triggered"})
        # else:
        #     return_array.append({"text": a, "ability_type": "static"})
    
    return return_array

def parse_scryfall_data(df: pd.DataFrame, magic_metadata_dict: dict) -> pd.DataFrame:
    df['is_split_card'] = df['name'].apply(lambda x: "//" in x)
    df['supertypes'] = df['type_line'].apply(get_supertypes_from_type_line)
    df['types'] = df['type_line'].apply(functools.partial(get_types_from_type_line, valid_card_types=magic_metadata_dict.get("type_and_subtype_info").keys()))
    df['subtypes'] = df['type_line'].apply(functools.partial(get_subtypes_from_type_line, type_to_subtype_dict=magic_metadata_dict.get("type_and_subtype_info")))
    df['oracle_text_mod'] = df.apply(functools.partial(parse_oracle_text, keywords=magic_metadata_dict.get("abilities").get("keywords")), axis=1)
    return df

def select_columns(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    return df[[x for x in columns if x in df.columns]]

def print_df(df: pd.DataFrame, rows: int) -> pd.DataFrame:
    for idx, row in [x for x in df.sample(frac=1).iterrows()][:rows]:
        print(f"{row['name']} - {" ".join(row['types'])}")
        print(row['keywords'])
        print(row['oracle_text'])
        print(row['oracle_text_mod'])
        print("\n")
    return df

commander_df = cards_df[cards_df.is_commander_legal & ~cards_df.oracle_text.isna() & cards_df.keywords.str.len().ge(3)].copy() \
    .pipe(drop_all_na_columns) \
    .pipe(rename_bool_cols) \
    .pipe(functools.partial(parse_scryfall_data, magic_metadata_dict=get_json_file("./data/magic_metadata.json"))) \
    .pipe(functools.partial(select_columns, columns=['oracle_id', 'name', 'types', 'oracle_text', 'oracle_text_mod', 'keywords'])) \
    .pipe(functools.partial(print_df, rows=2))

In [9]:
match: re.Match = re.search(r"^([a-z\s]+[,]*)+$", "first strike, vigilance")
match.groups()

(' vigilance',)